# Train2: fine-tune FunctionGemma on Colab

1. Upload `training/Train2/colab.zip` (made by `bash training/pack_colab.sh`) to the **root of My Drive**.
2. Runtime > Change runtime type > **GPU** (L4 or A100 preferred, T4 works but is slower).
3. Runtime > **Run all**. Allow access to Drive when asked.

When it finishes, the result is `MyDrive/train2_result.zip`: the trained model, the evaluation predictions and the logs.

- My Drive needs about **7 GB free** (two checkpoints plus the model, kept in `MyDrive/train2_out`).
- If Colab disconnects, reconnect and **Run all** again: training continues from the last finished epoch.
- A cell that fails stops Run all; read its error before running again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/fc && mkdir -p /content/fc && unzip -q /content/drive/MyDrive/colab.zip -d /content/fc
%cd /content/fc

In [ ]:
# keep Colab's torch
!pip install -q transformers==5.22.2 trl==1.17.1 datasets==5.1.0 accelerate==1.15.0

In [ ]:
!nvidia-smi

In [ ]:
%%bash
set -euo pipefail
cd /content/fc
# the output lives on Drive, so checkpoints survive a disconnect
mkdir -p /content/drive/MyDrive/train2_out
ln -sfn /content/drive/MyDrive/train2_out training/Train2/output
# under 20 GB (a T4 has 15 GB and no bf16): recompute activations to fit
mem=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -1)
if [ "$mem" -lt 20480 ]; then flag=--gradient-checkpointing; else flag=--no-gradient-checkpointing; fi
resume=
if ls -d training/Train2/output/functiongemma/checkpoint-* >/dev/null 2>&1; then
  resume=--resume
  echo "checkpoint found: resuming"
fi
python training/train.py $flag $resume 2>&1 | tee -a training/Train2/output/train.log
test -f training/Train2/output/functiongemma/model.safetensors

In [ ]:
%%bash
set -euo pipefail
cd /content/fc
test -f training/Train2/output/functiongemma/model.safetensors
python training/evaluate.py --models base,train2 --device cuda 2>&1 | tee training/Train2/output/eval.log

In [ ]:
%%bash
set -euo pipefail
cd /content/fc
test -f training/Train2/output/functiongemma/model.safetensors
rm -f /content/train2_result.zip
zip -r -q /content/train2_result.zip training/Train2/output/functiongemma training/Train2/output/eval_*.jsonl training/Train2/output/train.log training/Train2/output/eval.log -x 'training/Train2/output/functiongemma/checkpoint-*'
cp /content/train2_result.zip /content/drive/MyDrive/train2_result.zip
ls -lh /content/drive/MyDrive/train2_result.zip